# FINGUARD – Fraud Detection Analysis

Transaction-level preprocessing, exploratory analysis, machine-learning models, and evaluation.




In [ ]:
!pip install pandas


In [ ]:
import pandas as pd


In [ ]:

sample_df = df.sample(n=200000, random_state=42)

sample_df.to_csv("paysim_sample.csv", index=False)


In [ ]:
sample_df.shape


In [ ]:
sample_df.head()


In [ ]:
sample_df.info()


In [ ]:
#checking numm values
sample_df.isnull().sum()


In [ ]:
sample_df.drop_duplicates()


In [ ]:
sample_df[sample_df['amount'] > 0]


In [ ]:
#Step 5 — Fraud Distribution
sample_df['isFraud'].value_counts()


In [ ]:
sample_df['isFraud'].value_counts(normalize=True)*100


In [ ]:
#Step 6 — Transaction Type Analysis
sample_df['type'].value_counts()


In [ ]:
#Step 7 — Basic Statistics
sample_df['amount'].describe()


In [ ]:
!pip install matplotlib


In [ ]:
!pip install seaborn


In [ ]:
#Step 8 — Visualization
import matplotlib.pyplot as plt
import seaborn as sns
#Fraud distribution
sns.countplot(x='isFraud', data=sample_df)
plt.title("Fraud vs Non-Fraud Transactions")
plt.show()


In [ ]:
#fraud by transaction type
sns.countplot(x='type', hue='isFraud', data=sample_df)
plt.title("Fraud by Transaction Type")
plt.show()


In [ ]:
#Transaction Amount Distribution
import numpy as np
sns.histplot(np.log1p(sample_df['amount']), bins=50)
plt.title("Log Transformed Transaction Amount Distribution")
plt.xlabel("Log(Transaction Amount)")
plt.show()


In [ ]:
#Fraud vs Amount (Boxplot)
sns.boxplot(x='isFraud', y=np.log1p(sample_df['amount']), data=sample_df)

plt.title("Fraud vs Transaction Amount (Log Scale)")
plt.xlabel("Fraud (0 = Non-Fraud, 1 = Fraud)")
plt.ylabel("Log(Transaction Amount)")

plt.show()


In [ ]:
plt.figure(figsize=(10,6))
corr = sample_df.corr(numeric_only=True)
sns.heatmap(corr, annot=True, cmap='coolwarm')
plt.title("Feature Correlation Heatmap")
plt.show()


In [ ]:
# Safe feature engineering — no data leakage
sample_df['balanceDiffOrig'] = sample_df['oldbalanceOrg'] - sample_df['newbalanceOrig']
sample_df['balanceDiffDest'] = sample_df['newbalanceDest'] - sample_df['oldbalanceDest']
sample_df['amount_ratio']    = sample_df['amount'] / (sample_df['oldbalanceOrg'] + 1)
sample_df['amount_log']      = np.log1p(sample_df['amount'])


In [ ]:
# ---- STEP 1: Sort by time ----
sample_df = sample_df.sort_values(by='step')

# ---- STEP 2: Temporal Features ----

# 1. Transaction frequency (last 5 transactions)
sample_df['txn_count_rolling'] = (
    sample_df.groupby('nameOrig')['amount']
    .rolling(window=5, min_periods=1)
    .count()
    .reset_index(level=0, drop=True)
)

# 2. Rolling average amount
sample_df['avg_amt_rolling'] = (
    sample_df.groupby('nameOrig')['amount']
    .rolling(window=5, min_periods=1)
    .mean()
    .reset_index(level=0, drop=True)
)

# 3. Time gap between transactions
sample_df['time_diff'] = (
    sample_df.groupby('nameOrig')['step']
    .diff()
    .fillna(0)
)

# 4. Sudden spike indicator
sample_df['amount_spike'] = sample_df['amount'] / (sample_df['avg_amt_rolling'] + 1)

# ---- STEP 3: Handle NaN ----
sample_df.fillna(0, inplace=True)

print(sample_df[['txn_count_rolling','avg_amt_rolling','time_diff','amount_spike']].head())


In [ ]:
#TEMPORAL FRAUD VISUALIZATION

#A. Fraud count over time
import matplotlib.pyplot as plt

fraud_time = sample_df.groupby('step')['isFraud'].sum()

plt.figure(figsize=(10,5))
plt.plot(fraud_time)
plt.title("Fraud Occurrence Over Time")
plt.xlabel("Time Step")
plt.ylabel("Number of Fraud Transactions")
plt.show()

#B. Fraud rate over time
fraud_rate = sample_df.groupby('step')['isFraud'].mean()

# ---- Fraud Rate ----
fraud_rate = sample_df.groupby('step')['isFraud'].mean()

# ---- Smoothed version (PUT HERE) ----
fraud_rate_smooth = fraud_rate.rolling(window=20).mean()

plt.figure(figsize=(10,5))
plt.plot(fraud_rate_smooth)
plt.title("Smoothed Fraud Rate Over Time")
plt.xlabel("Time Step")
plt.ylabel("Fraud Probability")
plt.show()

#C. Burst fraud detection visualization
import numpy as np

fraud_time_diff = sample_df[sample_df['isFraud'] == 1]['time_diff']

# Keep small values also (not strict > 0)
fraud_time_diff = fraud_time_diff[fraud_time_diff >= 0]

plt.figure(figsize=(10,5))
plt.hist(np.log1p(fraud_time_diff), bins=50)
plt.title("Log Time Gap Distribution for Fraud")
plt.xlabel("Log(Time Difference)")
plt.ylabel("Frequency")
plt.show()


In [ ]:
# Drop leaky + unnecessary columns
cols_to_drop = ['errorBalanceOrig', 'errorBalanceDest', 
                'isFlaggedFraud', 'nameOrig', 'nameDest']

sample_df = sample_df.drop(columns=cols_to_drop, errors='ignore')


In [ ]:
# 'type' column already encoded in a previous run — skip get_dummies
# Just define X and y directly

X = sample_df.drop('isFraud', axis=1)
y = sample_df['isFraud']

print("Features used:", list(X.columns))
print("Shape:", X.shape)
print("Fraud %:", round(y.mean() * 100, 3))


In [ ]:
#Step 10 — Drop Unnecessary Columns
sample_df = sample_df.drop(columns=['nameOrig','nameDest','isFlaggedFraud'], errors='ignore')


In [ ]:
#Step 11 — Convert Transaction Type to Numeric
if 'type' in sample_df.columns:
    sample_df = pd.get_dummies(sample_df, columns=['type'], drop_first=True)


In [ ]:
#Step 12 — Define Features and Target
X = sample_df.drop('isFraud', axis=1)
y = sample_df['isFraud']


In [ ]:
!pip install scikit-learn


In [ ]:
#Step 13 — Train Test Split
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


In [ ]:
!pip install imbalanced-learn


In [ ]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)

X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)


In [ ]:
before = pd.Series(y_train).value_counts()
after = pd.Series(y_train_smote).value_counts()

comparison = pd.DataFrame({
    "Before SMOTE": before,
    "After SMOTE": after
})

print(comparison)


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# Count values
before_counts = pd.Series(y_train).value_counts().sort_index()
after_counts = pd.Series(y_train_smote).value_counts().sort_index()

labels = ["Non-Fraud", "Fraud"]
colors = ["green", "orange"]

plt.figure(figsize=(10,5))

# -------- Before SMOTE --------
plt.subplot(1,2,1)
bars1 = plt.bar(labels, before_counts, color=colors)
plt.title("Before SMOTE")
plt.ylabel("Number of Transactions")

for bar in bars1:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2,
             height,
             int(height),
             ha='center',
             va='bottom',
             fontsize=11)

# -------- After SMOTE --------
plt.subplot(1,2,2)
bars2 = plt.bar(labels, after_counts, color=colors)
plt.title("After SMOTE")

for bar in bars2:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2,
             height,
             int(height),
             ha='center',
             va='bottom',
             fontsize=11)

plt.tight_layout()
plt.show()


In [ ]:
# Step — Logistic Regression Model

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# Create model
lr = LogisticRegression(max_iter=1000)

# Train model using SMOTE data
lr.fit(X_train_smote, y_train_smote)

# Predict on test data
y_pred_lr = lr.predict(X_test)

# Accuracy
print("Accuracy:", accuracy_score(y_test, y_pred_lr))

# Confusion Matrix
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_lr))

# Classification Report
print("\nClassification Report:")
print(classification_report(y_test, y_pred_lr))


In [ ]:
#step-15 Decision Tree
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# Create model
dt = DecisionTreeClassifier(random_state=42)

# Train using SMOTE data
dt.fit(X_train_smote, y_train_smote)

# Predict
y_pred_dt = dt.predict(X_test)

# Accuracy
print("Decision Tree Accuracy:", accuracy_score(y_test, y_pred_dt))

# Confusion Matrix
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_dt))

# Classification Report
print("\nClassification Report:")
print(classification_report(y_test, y_pred_dt))


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import numpy as np



# ── Train Random Forest on noisy data ────────────────────────────
rf = RandomForestClassifier(
    n_estimators=100,      # reduced from 200
    max_depth=10,          # limit depth to prevent memorizing
    min_samples_leaf=5,    # minimum samples per leaf
    max_features='sqrt',   # use subset of features
    random_state=42
)
rf.fit(X_train_smote, y_train_smote)

# ── Predict on ORIGINAL test data (no noise) ─────────────────────
y_pred_rf  = rf.predict(X_test)
y_prob_rf  = rf.predict_proba(X_test)[:, 1]

# ── Results ───────────────────────────────────────────────────────
print("=" * 50)
print("  Random Forest Results")
print("=" * 50)
print(f"Accuracy: {accuracy_score(y_test, y_pred_rf):.4f}")
print(confusion_matrix(y_test, y_pred_rf))
print(classification_report(y_test, y_pred_rf,
      target_names=['Normal', 'Fraud']))


XGBoost
¶

In [ ]:
!pip install xgboost lightgbm


In [ ]:
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Train XGBoost
xgb = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    scale_pos_weight=(y_train == 0).sum() / (y_train == 1).sum(),
    random_state=42,
    eval_metric='logloss',
    verbosity=0
)
xgb.fit(X_train_smote, y_train_smote)

# Predict
y_pred_xgb  = xgb.predict(X_test)
y_prob_xgb  = xgb.predict_proba(X_test)[:, 1]

print("=" * 50)
print("  XGBoost Results")
print("=" * 50)
print(f"Accuracy: {accuracy_score(y_test, y_pred_xgb):.4f}")
print(classification_report(y_test, y_pred_xgb, target_names=['Normal', 'Fraud']))


LIGHTGBM
¶

In [ ]:
from lightgbm import LGBMClassifier

# Train LightGBM
lgbm = LGBMClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    class_weight='balanced',
    random_state=42,
    verbosity=-1
)
lgbm.fit(X_train_smote, y_train_smote)

# Predict
y_pred_lgbm = lgbm.predict(X_test)
y_prob_lgbm = lgbm.predict_proba(X_test)[:, 1]

print("=" * 50)
print("  LightGBM Results")
print("=" * 50)
print(f"Accuracy: {accuracy_score(y_test, y_pred_lgbm):.4f}")
print(classification_report(y_test, y_pred_lgbm, target_names=['Normal', 'Fraud']))


In [ ]:
from sklearn.metrics import roc_auc_score, average_precision_score, PrecisionRecallDisplay
import matplotlib.pyplot as plt

y_prob_lr = lr.predict_proba(X_test)[:, 1]
y_prob_dt = dt.predict_proba(X_test)[:, 1]
y_prob_rf = rf.predict_proba(X_test)[:, 1]

# Metrics table
metrics_df = pd.DataFrame({
    "Model":   ["Logistic Regression", "Decision Tree", "Random Forest"],
    "ROC-AUC": [roc_auc_score(y_test, y_prob_lr),
                roc_auc_score(y_test, y_prob_dt),
                roc_auc_score(y_test, y_prob_rf)],
    "PR-AUC":  [average_precision_score(y_test, y_prob_lr),
                average_precision_score(y_test, y_prob_dt),
                average_precision_score(y_test, y_prob_rf)],
})
print(metrics_df.to_string(index=False))

# Precision-Recall curve for all 3 models
fig, ax = plt.subplots(figsize=(8, 5))
PrecisionRecallDisplay.from_predictions(y_test, y_prob_lr, ax=ax, name="Logistic Regression")
PrecisionRecallDisplay.from_predictions(y_test, y_prob_dt, ax=ax, name="Decision Tree")
PrecisionRecallDisplay.from_predictions(y_test, y_prob_rf, ax=ax, name="Random Forest")
ax.set_title("Precision-Recall Curve — All Models")
plt.tight_layout()
plt.show()


In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Build comparison table
final_comparison = pd.DataFrame({
    "Model": ["Logistic Regression", "Decision Tree",
              "Random Forest", "XGBoost", "LightGBM"],
    "Accuracy":  [accuracy_score(y_test, y_pred_lr),
                  accuracy_score(y_test, y_pred_dt),
                  accuracy_score(y_test, y_pred_rf),
                  accuracy_score(y_test, y_pred_xgb),
                  accuracy_score(y_test, y_pred_lgbm)],
    "Precision": [precision_score(y_test, y_pred_lr),
                  precision_score(y_test, y_pred_dt),
                  precision_score(y_test, y_pred_rf),
                  precision_score(y_test, y_pred_xgb),
                  precision_score(y_test, y_pred_lgbm)],
    "Recall":    [recall_score(y_test, y_pred_lr),
                  recall_score(y_test, y_pred_dt),
                  recall_score(y_test, y_pred_rf),
                  recall_score(y_test, y_pred_xgb),
                  recall_score(y_test, y_pred_lgbm)],
    "F1 Score":  [f1_score(y_test, y_pred_lr),
                  f1_score(y_test, y_pred_dt),
                  f1_score(y_test, y_pred_rf),
                  f1_score(y_test, y_pred_xgb),
                  f1_score(y_test, y_pred_lgbm)],
    "ROC-AUC":   [roc_auc_score(y_test, y_prob_lr),
                  roc_auc_score(y_test, y_prob_dt),
                  roc_auc_score(y_test, y_prob_rf),
                  roc_auc_score(y_test, y_prob_xgb),
                  roc_auc_score(y_test, y_prob_lgbm)]
})

print(final_comparison.to_string(index=False))

# ── Plot ──────────────────────────────────────────────────────────
palette = ["#3d0066","#7b2d8b","#c0448a","#e06870","#f5b048"]

comparison_melt = final_comparison.melt(
    id_vars="Model", var_name="Metric", value_name="Score"
)
comparison_melt = comparison_melt[comparison_melt['Metric'] != 'Accuracy']

plt.figure(figsize=(14, 6))
ax = sns.barplot(
    data=comparison_melt,
    x="Model", y="Score", hue="Metric",
    palette="RdPu"
)
ax.set_facecolor("white")
plt.gcf().set_facecolor("white")
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['bottom'].set_color('#dddddd')
ax.spines['left'].set_color('#dddddd')
ax.tick_params(colors='#444444', labelsize=10)
ax.set_xticklabels(ax.get_xticklabels(), rotation=15)
ax.set_ylim(0, 1.1)
ax.yaxis.grid(True, color='#eeeeee', linewidth=0.8)
ax.set_axisbelow(True)

for container in ax.containers:
    ax.bar_label(container, fmt='%.2f', padding=3,
                 fontsize=7, color='#333333')

plt.title("All Models Comparison — Precision, Recall, F1, ROC-AUC",
          fontsize=14, fontweight='bold', color='#2d2d2d', pad=15)
plt.xlabel("Model", fontsize=11, color='#444444')
plt.ylabel("Score", fontsize=11, color='#444444')
plt.legend(title="Metric", fontsize=9,
           facecolor='white', edgecolor='#dddddd')
plt.tight_layout()
plt.show()


In [ ]:
#confusion matrix
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

cm = confusion_matrix(y_test, y_pred_xgb)

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.show()


In [ ]:
#ROC Curve
from sklearn.metrics import roc_curve, auc

y_prob = rf.predict_proba(X_test)[:,1]

fpr, tpr, _ = roc_curve(y_test, y_prob)

plt.plot(fpr, tpr)
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.show()


In [ ]:
#AUC Curve
from sklearn.metrics import roc_auc_score

auc = roc_auc_score(y_test, y_prob)
print("AUC Score:", auc)


In [ ]:
# Step — Feature Importance

import pandas as pd
import matplotlib.pyplot as plt

# Get feature importance
importance = rf.feature_importances_

# Create dataframe
feature_importance = pd.DataFrame({
    'Feature': X_train.columns,
    'Importance': importance
})

# Sort values
feature_importance = feature_importance.sort_values(by='Importance', ascending=False)

print(feature_importance)


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Get feature importance
importance = rf.feature_importances_

feature_importance = pd.DataFrame({
    "Feature": X_train.columns,
    "Importance": importance
})

# Sort values
feature_importance = feature_importance.sort_values(by="Importance", ascending=False)

# Select top 10 features
top_features = feature_importance.head(10)

# Plot
plt.figure(figsize=(10,6))

sns.barplot(
    x="Importance",
    y="Feature",
    data=top_features,
    palette="viridis"
)

plt.title("Top 10 Most Important Features for Fraud Detection", fontsize=14)
plt.xlabel("Importance Score")
plt.ylabel("Features")

plt.tight_layout()
plt.show()


IEEE-CIS DATASET
¶

In [ ]:
import pandas as pd


In [ ]:
train_trans=pd.read_csv("C:/Users/samatha p/Documents/train_transaction.csv")
train_id=pd.read_csv("C:/Users/samatha p/Documents/train_identity.csv")


In [ ]:
df_real = train_trans.merge(train_id, on="TransactionID", how="left")


In [ ]:
print(df_real.shape)


In [ ]:
df_fraud = df_real[df_real['isFraud'] == 1]
df_nonfraud = df_real[df_real['isFraud'] == 0]


In [ ]:
df_fraud_sample = df_fraud.sample(n=5000, random_state=42)
df_nonfraud_sample = df_nonfraud.sample(n=15000, random_state=42)

df_sample = pd.concat([df_fraud_sample, df_nonfraud_sample])
df_sample = df_sample.sample(frac=1, random_state=42)  # shuffle

print(df_sample.shape)


In [ ]:
features = [
    'TransactionAmt',
    'card1', 'card2', 'card3',
    'addr1', 'addr2',
    'dist1',
    'ProductCD_H', 'ProductCD_R', 'ProductCD_S', 'ProductCD_W'
]


In [ ]:
print(df_sample.columns.tolist())


In [ ]:
df_sample = df_sample[features + ['isFraud']]


In [ ]:
df_sample = df_sample.fillna(0)


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

sns.countplot(x='isFraud', data=df_sample)
plt.title("Fraud vs Non-Fraud Distribution")
plt.xlabel("Class (0 = Non-Fraud, 1 = Fraud)")
plt.ylabel("Count")
plt.show()


In [ ]:
plt.figure(figsize=(8,5))
sns.histplot(df_sample['TransactionAmt'], bins=50, kde=True)
plt.title("Transaction Amount Distribution")
plt.xlabel("Amount")
plt.show()


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Remove extreme values for better visualization
df_plot = df_sample[df_sample['TransactionAmt'] < 1000]
plt.figure(figsize=(8,5))
sns.boxplot(x='isFraud', y='TransactionAmt', data=df_plot)

plt.title("Transaction Amount vs Fraud (Cleaned View)")
plt.xlabel("Fraud (0 = No, 1 = Yes)")
plt.ylabel("Transaction Amount")
plt.show()


In [ ]:
plt.figure(figsize=(10,6))
sns.heatmap(df_sample.corr(), cmap='coolwarm', annot=False)
plt.title("Feature Correlation Heatmap")
plt.show()


In [ ]:
y_pred_real   # Random Forest
y_pred_lr     # Logistic Regression
y_pred_dt     # Decision Tree
y_pred_gb     # Gradient Boosting


In [ ]:
X = df_sample.drop('isFraud', axis=1)
y = df_sample['isFraud']


In [ ]:
from sklearn.model_selection import train_test_split

X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X, y, test_size=0.2, random_state=42
)


In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf_real = RandomForestClassifier(n_estimators=50, random_state=42)
rf_real.fit(X_train_r, y_train_r)


In [ ]:
from sklearn.metrics import classification_report

y_pred_real = rf_real.predict(X_test_r)

print("Real Dataset Performance:")
print(classification_report(y_test_r, y_pred_real))


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

lr = LogisticRegression(max_iter=1000)
lr.fit(X_train_r, y_train_r)

y_pred_lr = lr.predict(X_test_r)

print("Logistic Regression:")
print(classification_report(y_test_r, y_pred_lr))


In [ ]:
from sklearn.tree import DecisionTreeClassifier

dt = DecisionTreeClassifier(max_depth=5, random_state=42)
dt.fit(X_train_r, y_train_r)

y_pred_dt = dt.predict(X_test_r)

print("Decision Tree:")
print(classification_report(y_test_r, y_pred_dt))


In [ ]:
from sklearn.ensemble import GradientBoostingClassifier

gb = GradientBoostingClassifier(n_estimators=100, random_state=42)
gb.fit(X_train_r, y_train_r)

y_pred_gb = gb.predict(X_test_r)

print("Gradient Boosting:")
print(classification_report(y_test_r, y_pred_gb))


In [ ]:
from xgboost import XGBClassifier

xgb = XGBClassifier(use_label_encoder=False, eval_metric='logloss')
xgb.fit(X_train_r, y_train_r)

y_pred_xgb = xgb.predict(X_test_r)

print("XGBoost:")
print(classification_report(y_test_r, y_pred_xgb))


In [ ]:
from sklearn.metrics import accuracy_score

results = {
    "Random Forest": accuracy_score(y_test_r, y_pred_real),
    "Logistic Regression": accuracy_score(y_test_r, y_pred_lr),
    "Decision Tree": accuracy_score(y_test_r, y_pred_dt),
    "Gradient Boosting": accuracy_score(y_test_r, y_pred_gb),
    "XG Boost":accuracy_score(y_test_r,y_pred_xgb),
}

print(results)


In [ ]:
import matplotlib.pyplot as plt

model_names = list(results.keys())
accuracies = list(results.values())

plt.figure(figsize=(8,5))
plt.bar(model_names, accuracies)
plt.title("Model Comparison (Accuracy)")
plt.ylabel("Accuracy")
plt.xticks(rotation=30)
plt.show()


In [ ]:
#  ROC Curve (Model Performance)
from sklearn.metrics import roc_curve, auc
import matplotlib.pyplot as plt

# Get probability scores (VERY IMPORTANT for ROC)
y_prob = rf_real.predict_proba(X_test_r)[:, 1]

fpr, tpr, thresholds = roc_curve(y_test_r, y_prob)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(6,5))
plt.plot(fpr, tpr, label=f"AUC = {roc_auc:.3f}")
plt.plot([0,1], [0,1], linestyle='--')

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve - Fraud Detection")
plt.legend()
plt.show()


In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

cm = confusion_matrix(y_test_r, y_pred_real)

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title("Real Dataset Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()


In [ ]:
#Compare Multiple Models
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(),
    "Random Forest": RandomForestClassifier(n_estimators=50),
    "XG Boost": XGBClassifier(use_label_encoder=False, eval_metric='logloss'),
    "Gradient Boosting":GradientBoostingClassifier(n_estimators=100, random_state=42)
}

results = {}

for name, model in models.items():
    model.fit(X_train_r, y_train_r)
    y_pred = model.predict(X_test_r)
    acc = accuracy_score(y_test_r, y_pred)
    results[name] = acc

print(results)


In [ ]:
#Precision, Recall, F1
from sklearn.metrics import classification_report

for name, model in models.items():
    y_pred = model.predict(X_test_r)
    print(f"\n{name}")
    print(classification_report(y_test_r, y_pred))


In [ ]:
#Precision-Recall Curve (BETTER than ROC for fraud)
from sklearn.metrics import precision_recall_curve

y_prob = rf_real.predict_proba(X_test_r)[:,1]

precision, recall, _ = precision_recall_curve(y_test_r, y_prob)

plt.figure(figsize=(6,5))
plt.plot(recall, precision)
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curve")
plt.show()


In [ ]:
#Feature Importance
import pandas as pd

feat_importance = pd.Series(rf_real.feature_importances_, index=X_train_r.columns)
feat_importance.nlargest(10).plot(kind='barh')

plt.title("Top Features Influencing Fraud")
plt.show()
